# MMMPlotSuite v2 Migration Guide

PyMC-Marketing is moving from the monolithic `MMMPlotSuite` to a namespace-based plotting API.
The legacy suite will be removed in **pymc-marketing 2.0.0**.

This guide covers how to opt in to the new API and migrate your existing code.


## Opting In

Set `plot_suite = "new"` on an existing instance to switch to the new API:

```python
from pymc_marketing.mmm import MMM, TimeSliceCrossValidator

# MMM with new plot suite
mmm = MMM(...)
mmm.plot_suite = "new"

# Cross-validator with new plot suite
cv = TimeSliceCrossValidator(
    n_init=100,
    forecast_horizon=12,
    date_column="date",
)
cv.plot_suite = "new"
```


## `mmm.plot` namespace map

In the new API, `mmm.plot` returns a `MMMPlotSuiteFacade` with four sub-namespaces.

### Diagnostics

| Legacy | New |
|--------|-----|
| `mmm.plot.posterior_predictive(...)` | `mmm.plot.diagnostics.posterior_predictive(...)` |
| `mmm.plot.prior_predictive(...)` | `mmm.plot.diagnostics.prior_predictive(...)` |
| `mmm.plot.residuals_over_time(...)` | `mmm.plot.diagnostics.residuals_over_time(...)` |
| `mmm.plot.residuals_posterior_distribution(...)` | `mmm.plot.diagnostics.residuals_distribution(...)` |
| `mmm.plot.posterior_distribution(...)` | `mmm.plot.diagnostics.posterior(...)` |
| `mmm.plot.prior_vs_posterior(...)` | `mmm.plot.diagnostics.prior_vs_posterior(...)` |

### Decomposition

| Legacy | New |
|--------|-----|
| `mmm.plot.contributions_over_time(...)` | `mmm.plot.decomposition.contributions_over_time(...)` |
| `mmm.plot.waterfall_components_decomposition(...)` | `mmm.plot.decomposition.waterfall(...)` |
| `mmm.plot.channel_parameter(...)` | `mmm.plot.decomposition.channel_share_hdi(...)` |

### Sensitivity

| Legacy | New |
|--------|-----|
| `mmm.plot.sensitivity_analysis(...)` | `mmm.plot.sensitivity.analysis(...)` |
| `mmm.plot.uplift_curve(...)` | `mmm.plot.sensitivity.uplift(...)` |
| `mmm.plot.marginal_curve(...)` | `mmm.plot.sensitivity.marginal(...)` |

### Transformation

| Legacy | New |
|--------|-----|
| `mmm.plot.saturation_scatterplot(...)` | `mmm.plot.transformation.saturation_scatterplot(...)` |
| `mmm.plot.saturation_curves(...)` | `mmm.plot.transformation.saturation_curves(...)` |


## Budget plots

Budget plots have moved from `mmm.plot` to `optimizer.plot`.

```python
from pymc_marketing.mmm import BudgetOptimizerWrapper, MMM

mmm = MMM(...)
mmm.plot_suite = "new"
optimizer = BudgetOptimizerWrapper(model=mmm, start_date="2024-01-01", end_date="2024-12-31")

# Run optimization first to get samples
samples = optimizer.allocate_budget(...)

# New API
fig, axes = optimizer.plot.allocation_roas(samples=samples)
fig, axes = optimizer.plot.contribution_over_time(samples=samples)
```


## Cross-validation plots

CV plots use `MMMCVPlotSuite` when `plot_suite="new"`.

```python
cv = TimeSliceCrossValidator(
    n_init=100, forecast_horizon=12, date_column="date"
)
cv.plot_suite = "new"
cv_idata = cv.run(X, y, mmm=mmm)

# Parameter stability across folds
cv.plot.param_stability(cv_idata)

# Predictions vs actuals
cv.plot.predictions(cv_idata)

# CRPS score
cv.plot.crps(cv_idata)
```


## Removal timeline

The legacy `MMMPlotSuite` (accessed via `mmm.plot` when `mmm.plot_suite == "legacy"`, which is the default) will be **removed in pymc-marketing 2.0.0**.

Until then, using the legacy suite emits a `FutureWarning` on first access. To suppress it, opt in to the new API with `mmm.plot_suite = "new"`.
